# Programowanie i metody numeryczne

### Ćwiczenia 8.
### Aproksymacja funkcji II. Transformata Fouriera.

---

In [ ]:
import numpy as np

import scipy.interpolate
import scipy.fft

import matplotlib.pyplot as plt
%matplotlib inline

---

## Aproksymacja funkcji: interpolacja funkcjami sklejanymi

SciPy udostępnia narzędzia do interpolacji funkcjami sklejanymi w module `scipy.interpolate`.

Najważniejszym z nich jest funkcja `make_interp_spline` ([👉 dokumentacja i przykłady](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.make_interp_spline.html)), przeznaczona do interpolacji splajnami dowolnego stopnia w bazie B. Zwróć uwagę na parametr `k` opisujący stopień splajnu (w szczególności: `k = 1` dla splajnu liniowego, `k = 3` dla splajnu kubicznego; wartość `k = 3` jest domyślna) oraz parametr `bc_type` decydujący o tym, jakie warunki brzegowe zostaną narzucone (w przypadku splajnów stopnia większego od 1; np. warunki naturalne odpowiadają `bc_type = "natural"`).

Interpolację splajnem kubicznym można również przeprowadzić wykorzystując klasę `CubicSpline` ([👉 dokumentacja i przykłady](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html)). Korzystanie z niej jest zasadniczo równoważne użyciu funkcji `make_interp_spline` z parametrem `k = 3`. Różnica polega na tym, że `CubicSpline` stosuje bazę PP, a nie bazę B. Bywa to niekiedy użyteczne (np. ułatwia szukanie miejsc zerowych), jednak zasadniczo preferowana jest baza B.

Interpolacja trygonometryczna wymaga wykorzystania szybkiej transformaty Fouriera, dostępnej w module `scipy.fft`. Pomocne będą funkcje `fft` i `ifft` lub `rfft` i `irfft`. Uproszczony przykład takiej interpolacji znajdziesz [👉 tutaj](https://www.johndcook.com/blog/2024/11/05/trigonometric-interpolation/)

**Materiały dodatkowe**
* [SciPy User Guide: Interpolation](https://docs.scipy.org/doc/scipy/tutorial/interpolate.html)
  
  Przystępne i obszerne omówienie narzędzi do interpolacji udostępnianych przez SciPy. Z naszego punktu widzenia najbardziej przydatne informacje znajdują się w sekcjach [1-D interpolation](https://docs.scipy.org/doc/scipy/tutorial/interpolate/1D.html) oraz [Piecewise polynomials and splines](https://docs.scipy.org/doc/scipy/tutorial/interpolate/splines_and_polynomials.html).

* [SciPy API reference: Interpolation](https://docs.scipy.org/doc/scipy/reference/interpolate.html)
  
  Dokumentacja wszystkich narzędzi do interpolacji udostępnianych przez SciPy.

* [SciPy User Guide: Fourier Transforms](https://docs.scipy.org/doc/scipy/tutorial/fft.html)
  
  Przystępne i obszerne omówienie narzędzi do obliczania szybkiej transformaty Fouriera udostępnianych przez SciPy. Z naszego punktu widzenia najbardziej przydatne informacje znajdują się w sekcji [Fast Fourier transforms](https://docs.scipy.org/doc/scipy/tutorial/fft.html#fast-fourier-transforms).

* [SciPy API reference: Discrete Fourier transforms](https://docs.scipy.org/doc/scipy/reference/fft.html)
  
  Dokumentacja wszystkich narzędzi do obliczania szybkiej transformaty Fouriera udostępnianych przez SciPy.

### Zadanie 1. Jakość interpolacji wielomianowej i splajnowej.

Rozważmy trzy funkcje:
$$
f(x) = \sin x ,
\qquad
g(x) = \mathrm{e}^{-x^2} ,
\qquad
h(x) = \begin{cases}
x \sin \frac{1}{x} , & \text{gdy } x \neq 0 ,\\
0 , & \text{gdy } x = 0 .
\end{cases}

1. Zaimplementuj listę, której elementami będą te funkcje.

In [ ]:
functions = [
    lambda x: np.sin(x),
    lambda x: np.exp(- x**2),
    lambda x: np.piecewise(x, [x != 0], [lambda x: x * np.sin(1/x), 0])
]

2. Narysuj wykresy tych funkcji dla $x \in [-5, \, 5]$.

In [ ]:
x = np.linspace(-5, 5, 500)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax = ax.flatten()

for k, f in enumerate(functions):
    ax[k].plot(x, f(x))
    ax[k].set_title(f"y = {["f", "g", "h"][k]}(x)")
    ax[k].set_xlabel('x')
    ax[k].set_ylabel('y')
    ax[k].grid()
    ax[k].set_xlim(-5.2, 5.2)
    ax[k].set_ylim(-1.2, 1.2)

plt.tight_layout()


Porównamy teraz jakość interpolacji każdej z funkcji różnymi metodami, na zadanym przedziale $[a , \, b]$, dla zadanej liczby węzłów $N$. W każdym przypadku narysujemy wykresy interpolowanej funkcji, węzłów interpolacji, funkcji interpolującej oraz, na osobnym rysunku, błędu bezwzględnego interpolacji.

Będziemy eksperymentować z wartościami $a$, $b$ i $N$, dlatego najwygodniej będzie przechowywać je w zmiennych.

In [ ]:
a = 0
b = 5
N = 8

3. Dla każdej z funkcji przeprowadź interpolację wielomianem interpolacyjnym opartym na węzłach równoodległych.

In [ ]:
x = np.linspace(a, b, 500)

# Węzły interpolacji - równoodległe
q = np.linspace(a, b, N)

fig, ax = plt.subplots(2, 3, figsize=(12, 8), sharex='col', sharey='row')

for k, f in enumerate(functions):
    # Interpolacja wielomianowa Lagrange'a - węzły równoodległe
    w = scipy.interpolate.lagrange(q, f(q))

    # Błąd interpolacji
    err = np.abs(f(x) - w(x))

    # Wykresy
    ax[0, k].plot(x, f(x))
    ax[0, k].scatter(q, f(q), color='red')
    ax[0, k].plot(x, w(x))

    ax[0, k].set_title(f"y = {["f", "g", "h"][k]}(x)")
    ax[0, k].grid()

    ax[1, k].plot(x, err, color='green')
    ax[1, k].set_title(f"Błąd interpolacji {['f', 'g', 'h'][k]}(x)")
    ax[1, k].grid()

plt.tight_layout()

4. Dla każdej z funkcji przeprowadź interpolację wielomianem interpolacyjnym opartym na węzłach Czebyszewa.

In [ ]:
x = np.linspace(a, b, 500)

# Węzły interpolacji - Czebyszewa
q = 0.5 * (a + b) + 0.5 * (b - a) * np.cos((2 * np.arange(1, N + 1) - 1) / (2 * N) * np.pi)

fig, ax = plt.subplots(2, 3, figsize=(12, 8), sharex='col', sharey='row')

for k, f in enumerate(functions):
    # Interpolacja wielomianowa Lagrange'a - węzły Czebyszewa
    w = scipy.interpolate.lagrange(q, f(q))

    # Błąd interpolacji
    err = np.abs(f(x) - w(x))

    # Wykresy
    ax[0, k].plot(x, f(x))
    ax[0, k].scatter(q, f(q), color='red')
    ax[0, k].plot(x, w(x))

    ax[0, k].set_title(f"y = {["f", "g", "h"][k]}(x)")
    ax[0, k].grid()

    ax[1, k].plot(x, err, color='green')
    ax[1, k].set_title(f"Błąd interpolacji {['f', 'g', 'h'][k]}(x)")
    ax[1, k].grid()

plt.tight_layout()

5. Dla każdej z funkcji przeprowadź interpolację liniowym splajnem interpolacyjnym opartym na węzłach równoodległych.

In [ ]:
x = np.linspace(a, b, 500)

# Węzły interpolacji - równoodległe
q = np.linspace(a, b, N)

fig, ax = plt.subplots(2, 3, figsize=(12, 8), sharex='col', sharey='row')

for k, f in enumerate(functions):
    # Interpolacja splajnem liniowym (k = 1) - węzły równoodległe
    w = # *** Miejsce na Twoje rozwiązanie ************************************

    # Błąd interpolacji
    err = np.abs(f(x) - w(x))

    # Wykresy
    ax[0, k].plot(x, f(x))
    ax[0, k].scatter(q, f(q), color='red')
    ax[0, k].plot(x, w(x))

    ax[0, k].set_title(f"y = {["f", "g", "h"][k]}(x)")
    ax[0, k].grid()

    ax[1, k].plot(x, err, color='green')
    ax[1, k].set_title(f"Błąd interpolacji {['f', 'g', 'h'][k]}(x)")
    ax[1, k].grid()

plt.tight_layout()

6. Dla każdej z funkcji przeprowadź interpolację naturalnym kubicznym splajnem interpolacyjnym opartym na węzłach równoodległych.

In [ ]:
x = np.linspace(a, b, 500)

# Węzły interpolacji - równoodległe
q = np.linspace(a, b, N)

fig, ax = plt.subplots(2, 3, figsize=(12, 8), sharex='col', sharey='row')

for k, f in enumerate(functions):
    # Interpolacja naturalnym splajnem kubicznym (k = 3, warunki brzegowe naturalne) - węzły równoodległe
    w = # *** Miejsce na Twoje rozwiązanie ************************************

    # Błąd interpolacji
    err = np.abs(f(x) - w(x))

    # Wykresy
    ax[0, k].plot(x, f(x))
    ax[0, k].scatter(q, f(q), color='red')
    ax[0, k].plot(x, w(x))

    ax[0, k].set_title(f"y = {["f", "g", "h"][k]}(x)")
    ax[0, k].grid()

    ax[1, k].plot(x, err, color='green')
    ax[1, k].set_title(f"Błąd interpolacji {['f', 'g', 'h'][k]}(x)")
    ax[1, k].grid()

plt.tight_layout()

Wykonaj punkty od 3. do 6. wielokrotnie, zmieniając wartości $a$, $b$ i $N$ (pamiętaj, że są one określone tuż nad punktem 3.). Jak jakość interpolacji zależy od tych parametrów?

### Zadanie 2. Aproksymacja danych pomiarowych.

*Wykorzystywane w zadaniu dane zostały udostępnione przez Instytut Meteorologii i Gospodarki Wodnej
Państwowy Instytut Badawczy. Należy z nich korzystać zgodnie z regulaminem udostępniania danych, dostępnym na stronie Instytutu.*

Plik `temperatura_Bielany_2025-03.csv` zawiera wyniki pomiarów temperatury powietrza dokonywanych w stacji meteorologicznej Warszawa-Bielany w marcu 2025 roku. Pomiary były wykonywane co 10 minut. Dane przedstawione są w formacie CSV: pierwsza kolumna zawiera datę i godzinę pomiaru, druga - temperaturę powietrza wyrażoną w stopniach Celsjusza.

Naszym celem będzie porównanie jakości różnych metod interpolacji tych danych: wielomianowej, trygonometrycznej, liniową funkcją sklejaną oraz kubiczną funkcją sklejaną. Spośród wszystkich punktów pomiarowych wybierzemy tylko niektóre, odpowiadające pełnym godzinom, a następnie obliczymy przybliżone wartości temperatury powietrza w pozostałych punktach, posługując się każdą z wymienionych metod interpolacji, i porównamy je z rzeczywistymi, zmierzonymi wartościami.

1. Narysuj wykres punktowy danych. Oś odciętych powinna przedstawiać datę i godzinę pomiaru, natomiast oś rzędnych - wynik pomiaru. Zadbaj o to, by etykiety na osiach były czytelne.

In [ ]:
# Miejsce na Twoje rozwiązanie

Można przypuszczać, że wartości temperatury wykazują pewną okresowość. Czy rzeczywiście tak jest? Jeśli tak, po której z metod interpolacji można oczekiwać najlepszych, a po której najgorszych wyników?

2. Wybierając spośród wszystkich pomiarów tylko te dokonywane co $K$ godzin, gdzie $K$ jest zadaną liczbą naturalną, i traktując je jako węzły interpolacji, wyznacz funkcje interpolujące dla każdej z wymienionych wyżej metod interpolacji.

In [ ]:
K = 1

In [ ]:
# Interpolacja wielomianowa

# Miejsce na Twoje rozwiązanie

In [ ]:
# Interpolacja trygonometryczna

# Miejsce na Twoje rozwiązanie

In [ ]:
# Interpolacja splajnem liniowym

# Miejsce na Twoje rozwiązanie

In [ ]:
# Interpolacja naturalnym splajnem kubicznym

# Miejsce na Twoje rozwiązanie

3. Dla każdej z wykorzystanych metod interpolacji sporządź dwa wykresy. Na pierwszym mają się znaleźć punkty pomiarowe, przy czym te z nich, które były węzłami interpolacji, muszą być w jakiś sposób wyróżnione (np. kolorem i/lub rozmiarem), oraz wykres funkcji interpolującej. Drugi z wykresów ma przedstawiać błąd interpolacji - moduł różnicy pomiędzy rzeczywistym wynikiem pomiaru a wartością funkcji interpolującej dla wszystkich punktów pomiarów. Przygotuj także inforację o czasie wyznaczania funkcji interpolacyjnej (może ją np. umieścić we wspólnym tytule obu wykresów, wraz z nazwą metody interpolacji).

In [ ]:
# Miejsce na Twoje rozwiązanie

Które z metod interpolacji okazały się najskuteczniejsze? Jak zależy jakość interpolacji każdą z metod od $K$?

### Zadanie 3. Aproksymacja funkcji nieciągłych.

Rozważmy funkcje:
\begin{align*}
f : \, \left[ -1 , \, 1 \right] \to \mathbb{R} : \qquad & f(x) = \begin{cases}
0 , & \text{gdy } x \in \left[ -1 , \, 0 \right[ , \\
1 , & \text{gdy } x \in \left[ 0 , \, 1 \right] ,
\end{cases} \\
g : \, \left[ -1 , \, 1 \right] \to \mathbb{R} : \qquad & g(x) = \begin{cases}
    1 , & \text{gdy } x = 0 , \\
    0 , & \text{w przeciwnym przypadku} , 
\end{cases}
\end{align*}

1. Zaimplementuj te funkcje. Wykorzystaj `numpy.piecewise`.

In [ ]:
def f(x: float) -> float:
    return # Miejsce na Twoje rozwiązanie

In [ ]:
def g(x: float) -> float:
    return # Miejsce na Twoje rozwiązanie

2. Wybierając $N$ równoodległych liczb z przedziału $[-1 , \, 1]$ i traktując je jako węzły interpolacji, wyznacz funkcję interpolacyjną dla funkcji $f$ i $g$ w przypadku interpolacji wielomianowej, trygonometrycznej, liniową funkcją sklejaną oraz kubiczną funkcją sklejaną.

In [ ]:
N = 8

In [ ]:
# Funkcja f

# Miejsce na Twoje rozwiązanie

In [ ]:
# Funkcja g

# Miejsce na Twoje rozwiązanie

3. Dla każdej z funkcji $f$ i $g$ oraz dla każdej z wykorzystanych metod interpolacji narysuj dwa wykresy. Na pierwszym mają się znaleźć wykres funkcji interpolowanej, węzły interpolacji oraz wykres funkcji interpolującej. Drugi z wykresów ma przedstawiać błąd interpolacji -- moduł różnicy pomiędzy wartością funkcji interpolowanej a wartością funkcji interpolującej. Program powinien także zwracać informację o czasie wyznaczania funkcji interpolacyjnej (może ją np. umieścić we wspólnym tytule obu wykresów, wraz z nazwą metody interpolacji).

In [ ]:
# Funkcja f

# Miejsce na Twoje rozwiązanie

In [ ]:
# Funkcja g

# Miejsce na Twoje rozwiązanie

Która z metod interpolacji okazała się najskuteczniejsza w przypadku każdej z funkcji? Czy zależy to od wartości $N$?

---

## Transformata Fouriera

SciPy udostępnia narzędzia do obliczania transformaty Fouriera w module `scipy.fft`.

Nie należy używać modułów `scipy.fftpack` oraz `numpy.fft` - są one przestarzałe, nie zostały usuniętę w ramach zachowania kompatybilności wstecznej.

**Materiały dodatkowe**
* [SciPy User Guide: Fourier Transforms](https://docs.scipy.org/doc/scipy/tutorial/fft.html)
  
  Przystępne i obszerne omówienie narzędzi do obliczania szybkiej transformaty Fouriera udostępnianych przez SciPy. Z naszego punktu widzenia najbardziej przydatne informacje znajdują się w sekcji [Fast Fourier transforms](https://docs.scipy.org/doc/scipy/tutorial/fft.html#fast-fourier-transforms).

* [SciPy API reference: Discrete Fourier transforms](https://docs.scipy.org/doc/scipy/reference/fft.html)
  
  Dokumentacja wszystkich narzędzi do obliczania szybkiej transformaty Fouriera udostępnianych przez SciPy.

### Zadanie 4. Usuwanie sumu.

Przeanalizuj przykłady i odtwórz wyniki zaprezentowanej tutaj:

<https://github.com/arif-du/Digital-Signal-Processing-with-Python/blob/main/15_DFT_filtering.ipynb>

---

## Praca domowa

### Zadanie 5. Zachorowania na COVID-19.

Wrócimy do zadania domowego z poprzednich ćwiczeń, tym razem wykorzystując funkcje sklejane.

Napisz kod wykreślający krzywą zachorowań na COVID-19 dla wybranej przez użytkownika lokalizacji. 

Twój kod powinien pobierać z internetu plik z danymi dotyczącymi tygodniowej liczby zachorowań w formacie CSV, udostępniony pod adresem

https://opendata.ecdc.europa.eu/covid19/nationalcasedeath/csv/data.csv

a następnie wypisywać listę lokalizacji, dla których dane są dostępne, prosić użytkownika o wskazanie jednej z nich i rysować krzywą przedstawiającą dzienną ilość nowych zachorowań na COVID--19 w tej lokalizacji.

W celu uzyskania informacji o dziennej liczbie zachorowań na podstawie dostępnych danych posłuż się interpolacją splajnem kubicznym oraz naturalnym splajnem liniowym.

In [ ]:
# Interpolacja splajnem liniowym

# Miejsce na Twoje rozwiązanie

In [ ]:
# Interpolacja naturalnym splajnem kubicznym

# Miejsce na Twoje rozwiązanie

Jak bardzo różnią się wyniki? Narysuj wykres różnic wartości obu interpolantów.

In [ ]:
# Miejsce na Twoje rozwiązanie

Porównaj wizualnie obie otrzymane na początku krzywe do krzywej, którą uzyskałem w pracy domowej z poprzednich ćwiczeń? Czy poprawa jakości interpolacji jest zauważalna?